# TUGAS MANDIRI — PERTEMUAN 6
### Format Data Besar (Parquet & JSON) dan Pipeline ETL Sederhana

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 6 |
| **Topik** | Parquet, JSON, Partitioning, & Pipeline ETL |
| **Nama Mahasiswa** | *(Isi Nama Anda)* |
| **NPM** | *(Isi NPM Anda)* |

---
## Skenario Tugas
Membangun pipeline ETL (*Extract, Transform, Load*) skala produksi yang menggabungkan 3 sumber data dengan format berbeda: Transaksi (CSV), Master Produk (JSON Lines), dan Ulasan Pelanggan (CSV).


### Persiapan Dataset (3 Sumber Data)


In [ ]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# 1. Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# 2. Data Produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000, 50000, 75000, 100000, 150000, 250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# 3. Data Ulasan (CSV) — 3500 baris
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print("Tiga sumber data berhasil dibuat.")


### Bagian A: EXTRACT — Membaca Ketiga Format Data


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, round as spark_round

spark = SparkSession.builder \
    .appName("Tugas6_ETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Transaksi: {df_transaksi.count()} baris")
df_transaksi.printSchema()

print(f"Produk: {df_produk.count()} baris")
df_produk.printSchema()

print(f"Ulasan: {df_ulasan.count()} baris")
df_ulasan.printSchema()


### Bagian B: TRANSFORM — Penggabungan (Join) & Pendapatan


In [ ]:
# Left join ke ulasan (karena tidak semua transaksi ada ulasan)
# Inner join ke produk (karena setiap transaksi pasti punya produk valid)
df_etl = df_transaksi.join(df_ulasan, on="order_id", how="left") \
                     .join(df_produk, on="product_id", how="inner") \
                     .withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_etl.show(5)


### Bagian C: TRANSFORM — Penanganan Data Kosong & Flag `ada_ulasan`


In [ ]:
# 1. Menambahkan kolom ada_ulasan sebelum imputasi null
df_etl = df_etl.withColumn("ada_ulasan", col("rating").isNotNull())

# 2. Imputasi nilai rating kosong menjadi 0
# Alasan: Nilai 0 secara logis membedakan transaksi yang memang belum diulas (0) 
# dengan transaksi yang diulas buruk oleh pelanggan (rating 1).
df_etl = df_etl.na.fill({"rating": 0})

df_etl.select("order_id", "nama_produk", "kategori", "total_pendapatan", "rating", "ada_ulasan").show(10)


### Bagian D: LOAD — Menyimpan ke Parquet Terpartisi


In [ ]:
output_parquet = "tugas6_hasil_etl_parquet"

# Menyimpan ke Parquet dipartisi per kategori
df_etl.write.mode("overwrite").partitionBy("kategori").parquet(output_parquet)

# Verifikasi struktur partisi
!ls -la {output_parquet}

# Membaca kembali dari Parquet untuk verifikasi keutuhan
df_baca_kembali = spark.read.parquet(output_parquet)
print(f"Verifikasi total baris terbaca: {df_baca_kembali.count()} (Harus sama dengan {df_transaksi.count()})")


### Bagian E: Business Insight & Interpretasi


In [ ]:
# Analisis persentase transaksi yang memiliki ulasan per kategori
insight_ulasan = df_etl.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_round((spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)) / count("order_id")) * 100, 2).alias("persen_ada_ulasan")
).orderBy("persen_ada_ulasan")

insight_ulasan.show()


#### Interpretasi Bisnis:
Kategori produk dengan persentase transaksi berulasan paling rendah merupakan sinyal kritis bagi tim marketing dan *customer experience*. Minimnya feedback pada kategori tersebut menyebabkan perusahaan kesulitan mengevaluasi kepuasan pembeli dan kualitas produk. Tim marketing dapat merancang kampanye insentif ulasan (seperti cashback atau koin loyalitas khusus pada kategori tersebut) guna mendorong partisipasi pelanggan dan meningkatkan konversi penjualan jangka panjang.
